# NLA Steering — GPU runner

Единственный ноутбук проекта. Запусти ячейки сверху вниз **один раз** за сессию
и оставь последнюю работать: она забирает задачи из очереди на Drive,
выполняет их и складывает результаты обратно.

Runtime → Change runtime type → **T4 GPU**.

Что нужно один раз настроить в панели слева 🔑 **Secrets** (и включить доступ
этому ноутбуку для каждого):

| секрет | зачем | где взять |
|---|---|---|
| `HF_TOKEN` | скачивание весов | [hf.co/settings/tokens](https://huggingface.co/settings/tokens) |
| `GH_TOKEN` | клонирование приватного репозитория | [fine-grained PAT](https://github.com/settings/personal-access-tokens/new), доступ к `xatxor/NLA_Steering_DLS`, Contents: Read-only |

`GH_TOKEN` не нужен, если сделать репозиторий публичным.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, subprocess, sys
from pathlib import Path
from google.colab import userdata

REPO_URL = "https://github.com/xatxor/NLA_Steering_DLS.git"
REPO = Path('/content/repo')


def secret(name):
    """Colab Secrets; отсутствующий или невыданный секрет — не ошибка."""
    try:
        return userdata.get(name)
    except Exception:
        return None


def sh(*args, cwd=None):
    """subprocess с внятной ошибкой: git пишет причину в stderr, и её нужно видеть."""
    p = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if p.returncode:
        print(p.stdout, p.stderr, sep='\n')
        raise RuntimeError(f'{" ".join(args)} -> exit {p.returncode}')
    return p.stdout.strip()


# Репозиторий приватный, поэтому нужен GH_TOKEN (fine-grained PAT с правом
# Contents: Read). Токен отдаётся гиту через GIT_ASKPASS, а не через URL, —
# так он не оседает в .git/config и не печатается в выводе ячейки.
gh_token = secret('GH_TOKEN')
if gh_token:
    askpass = Path('/content/.git_askpass.sh')
    askpass.write_text('#!/bin/sh\nprintf %s "$GH_TOKEN"\n')
    askpass.chmod(0o700)
    os.environ['GH_TOKEN'] = gh_token
    os.environ['GIT_ASKPASS'] = str(askpass)
os.environ['GIT_TERMINAL_PROMPT'] = '0'  # не виснуть, ожидая ввод логина

try:
    if (REPO / '.git').exists():
        sh('git', '-C', str(REPO), 'fetch', 'origin')
        sh('git', '-C', str(REPO), 'reset', '--hard', 'origin/main')
    else:
        sh('git', 'clone', REPO_URL, str(REPO))
except RuntimeError:
    print('\n' + '=' * 70)
    print('Не удалось получить репозиторий. Он приватный, значит нужен доступ:')
    print('  1. https://github.com/settings/personal-access-tokens/new')
    print('     Repository access -> xatxor/NLA_Steering_DLS, Contents: Read-only')
    print('  2. Colab, панель слева 🔑 -> добавить секрет GH_TOKEN и включить')
    print('     для него доступ этому ноутбуку')
    print('  3. перезапустить эту ячейку')
    print('Альтернатива: сделать репозиторий публичным, тогда токен не нужен.')
    print('=' * 70)
    raise

os.environ['HF_TOKEN'] = secret('HF_TOKEN') or ''
if not os.environ['HF_TOKEN']:
    print('! HF_TOKEN не найден в Colab Secrets — скачивание весов упрётся в лимиты')

# Кэш HF на локальный диск инстанса, а не на Drive: Drive медленный на
# множестве мелких файлов и его квота нам дороже, чем эфемерные 100 ГБ Colab.
os.environ['HF_HOME'] = '/content/hf_cache'

sh(sys.executable, '-m', 'pip', 'install', '-q',
   'transformers>=4.46', 'accelerate>=1.0', 'datasets>=3.0',
   'sentence-transformers>=3.0', 'python-dotenv', 'pyyaml', 'safetensors')

sys.path.insert(0, str(REPO / 'src'))
from nla_steering.paths import ensure_layout, workspace
ensure_layout()
print('репо   :', sh('git', '-C', str(REPO), 'log', '--oneline', '-1'))
print('рабочая:', workspace())


In [ ]:
import torch

print('torch      :', torch.__version__)
print('cuda       :', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    free, _ = torch.cuda.mem_get_info()
    print('gpu        :', props.name)
    print('память     : %.2f ГБ, свободно %.2f ГБ' % (props.total_memory / 1024**3,
                                                      free / 1024**3))
    print('capability :', f'{props.major}.{props.minor}')
    # Осторожно: is_bf16_supported() учитывает программную эмуляцию и на Turing
    # возвращает True. Нативная поддержка (тензорные ядра) — с Ampere, 8.0+.
    native_bf16 = (props.major, props.minor) >= (8, 0)
    print('bf16       : эмуляция=%s, нативно=%s -> считаем в fp16'
          % (torch.cuda.is_bf16_supported(), native_bf16))


In [ ]:
import subprocess, time, traceback
from datetime import datetime, timezone

from nla_steering import jobs

POLL = 15  # секунд


def refresh_secrets():
    """Перечитать секреты перед каждой задачей.

    Воркер занимает ядро своим циклом, поэтому ячейку настройки нельзя
    перезапустить, не остановив его. Без этого обновлённый в панели Colab токен
    не доезжает до задач: процесс держит значение, прочитанное при старте.
    """
    changed = []
    for name in ('HF_TOKEN', 'GH_TOKEN'):
        value = (secret(name) or '').strip()
        if value and os.environ.get(name) != value:
            os.environ[name] = value
            changed.append(name)
    return changed


def run_job(job, job_path):
    log = jobs.log_path(job.id)
    with log.open('w', encoding='utf-8', buffering=1) as fh:
        def emit(line):
            print(line, end='')
            fh.write(line)

        emit(f'=== {job.id}\n{job.note}\n{job.command()}\n\n')

        updated = refresh_secrets()
        if updated:
            emit(f'секреты обновлены: {", ".join(updated)}\n')

        # Подтягиваем код перед каждой задачей: я пушу правку и сразу ставлю job,
        # так что воркер должен работать по свежему коммиту. Провал синхронизации
        # не молчаливый — иначе задача незаметно уйдёт на старом коде.
        sync = subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'],
                              capture_output=True, text=True)
        if sync.returncode == 0:
            sync = subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'],
                                  capture_output=True, text=True)
        if sync.returncode:
            emit(f'! git sync не удался, работаем по локальному коду\n{sync.stderr}\n')
        emit(f'commit: {subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"], capture_output=True, text=True).stdout}\n')

        proc = subprocess.Popen(
            [sys.executable, job.script, *job.args],
            cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
            env={**os.environ, 'PYTHONPATH': str(REPO / 'src'), 'PYTHONUNBUFFERED': '1'},
        )
        for line in proc.stdout:
            emit(line)
        code = proc.wait()
        emit(f'\n=== exit {code}\n')
    jobs.finish(job_path, ok=(code == 0))
    return code


# Задача, которую забрал умерший воркер, остаётся в running/ навсегда:
# новый воркер её не тронет. Обрывы Colab — это норма (потеря связи, лимит
# сессии), поэтому при старте возвращаем зависшие задачи в очередь.
recovered = jobs.requeue_stale(older_than=900)
if recovered:
    print(f'возвращены в очередь после обрыва: {", ".join(recovered)}')

print('воркер запущен, жду задачи. Останов — кнопка stop.')
idle_since = time.monotonic()
while True:
    try:
        claimed = jobs.claim()
        if claimed is None:
            if time.monotonic() - idle_since > 300:
                print(f'{datetime.now(timezone.utc):%H:%M:%S} простой, очередь пуста')
                idle_since = time.monotonic()
            time.sleep(POLL)
            continue
        idle_since = time.monotonic()
        job, job_path = claimed
        print(f'\n>>> {job.id}')
        code = run_job(job, job_path)
        print(f'<<< {job.id} exit={code}')
    except KeyboardInterrupt:
        print('остановлен')
        break
    except Exception:
        traceback.print_exc()
        time.sleep(POLL)
